# A1.11 · Rogue agents in a multi-agent system

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.10 · Agent communication poisoning](https://spbreed.github.io/cyber-commons/lessons/A1.10.html)**.

| | |
|---|---|
| Tools used | SPIFFE/SPIRE, kagent |

## What this lesson is

**What it covers.** Introduce an unregistered agent into the topology and have it receive delegated work.

**Why a security engineer needs it.** An agent nobody approved receives delegated work and delegated authority, and the orchestrator has no way to tell it apart from a legitimate worker. The control it builds is: a registry of approved agents with identity-bound admission (A2.5) and an audit trail per hop (A2.7).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

The orchestrator delegates to whatever agents it discovers. Something joined the pool this morning that nobody registered, and it has been receiving work ever since, with the same standing as the agents you wrote.

> **At CyberTravels.** CyberTravels' orchestrator delegates to the agents it discovers. A fifth one joined the pool during a deployment last week and has been receiving bookings ever since.

## 2 · The framework

```
   registered           discovered at runtime
   +---------+          +---------+   +---------+
   | agent A |          | agent B |   |   ???   |  <- joined this morning
   +----+----+          +----+----+   +----+----+
        |                    |             |
        +-------- orchestrator delegates ---+

   the pool is a trust boundary. most orchestrators treat it as a config file.
```

**OWASP T13 — Rogue Agents in Multi-Agent Systems.**

The **orchestrator** delegates work to agents. The question this risk asks is
disarmingly simple: *how does it know which agents are allowed to receive that
work?*

In most deployments the answer is configuration — a list in a file, an env var,
a service discovery lookup. None of those is an identity check. An agent that
appears in the right place, answering the right protocol, is treated as a
legitimate worker.

Two ways one arrives:

**A compromised legitimate agent.** It was registered and approved; it is now
executing someone else's instructions after A1.3 or A1.10. Nothing about its
registration is wrong, which is why registration alone does not solve this.

**An unregistered agent.** A developer stood one up to test something, or an
attacker with a foothold registered a service. It receives delegated work and
delegated authority because the topology admits by convention rather than by
identity.

The consequence specific to multi-agent systems: **delegated authority flows to
it.** The orchestrator does not just send a task, it sends the context and often
a token. So an agent nobody approved ends up holding a credential that narrows
from a real user's, and the audit trail — if it records agent names at all —
records a name the attacker chose.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

An orchestrator that admits workers by configuration.

## 4 · The check, as a skill

CyberTravels discovers its workers. The skill compares what is present against what anybody registered, then follows a delegation to see what the unregistered agent is handed — including the narrowed traveller token.

In [ ]:
# skills/threats/agent-registry-gap-check/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: agent-registry-gap-check
description: >-
  Compare the agents actually present in an environment against the agents
  anybody registered, and establish what work and which tokens the unregistered
  ones are handed. Use when agents are created programmatically, when a
  discovery protocol is in play, or when asked how a worker joins a fleet.
allowed-tools: Read, Grep, Glob
---

# Discovery is not admission

A topology that discovers its workers will discover whatever answers. The
defect is not that an unknown agent exists — it is that the orchestrator hands
it the same delegated work, and the same narrowed user token, as a registered
one. The unregistered agent then acts **as the user** against anything
downstream that honours the token.

## When to use this

Any fleet where agents register themselves, are discovered over a network, or
are spawned by other agents. Also after adding a marketplace, a plugin
mechanism or an A2A-style protocol.

## Procedure

**1 — Enumerate what is present.** Ask the runtime, not the design document:
running workloads, connected clients, queue consumers, whatever the discovery
mechanism itself returns.

**2 — Enumerate what is registered.** The list somebody maintains, with owners.
Both lists in hand, the gap is arithmetic.

**3 — Follow a delegation.** For one task, record which agents received work
and what credential travelled with it. The question is whether registration is
consulted *before* delegation or only reported afterwards.

**4 — Establish what the token permits.** A narrowed on-behalf-of token in the
hands of an unregistered agent is a user session with no login. Name the
downstreams that would honour it.

**5 — Separate admission from identity.** Registration answers "should this
agent exist"; workload identity answers "is this the agent it claims to be".
Report which of the two is missing, because they are different projects.

## Output contract

```json
{
  "present": ["str"],
  "registered": ["str"],
  "gap": ["str"],
  "delegations": [{"agent": "str", "registered": false, "credential": "str"}],
  "token": {"kind": "str", "acts_as_user": true, "honoured_by": ["str"]},
  "missing": {"admission": true, "workload_identity": true}
}
```

## Failure modes

- **Reading the registry as the inventory.** The registry is the claim; the
  runtime is the fact.
- **Treating discovery as authentication.** Answering is not proving.
- **Reporting the unregistered agent as the problem.** The problem is that
  delegation never consulted the registry.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, os, shutil, sys

# Make the shared runtime importable, then import it. On Kaggle an attached
# kernel is mounted as __script__.py — not on sys.path and not named after the
# kernel — so copy it to the name it is imported by. Locally it is already a
# file of that name in the repository.
_k = glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py", recursive=True)
if _k:
    shutil.copy(_k[0], "cyber_commons_skill_runtime.py")
sys.path[:0] = [".", "skills/_runtime", "../skills/_runtime", "../../skills/_runtime"]

from cyber_commons_skill_runtime import run_skill

# Split skills/threats/agent-registry-gap-check/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/threats/agent-registry-gap-check/scripts/agent_registry_gap_check.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Compare the agents that exist with the agents anybody registered, and show what the unregistered ones are handed.

This is the executable half of the `agent-registry-gap-check` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

REGISTRY = {"pricing-agent":  {"owner": "payments-team", "approved": True},
            "billing-agent":  {"owner": "payments-team", "approved": True}}

DISCOVERED = ["pricing-agent", "billing-agent", "reporting-agent-v2"]

DELEGATED = []

def delegate(agent_name, task, user_token):
    """The orchestrator hands work - and the caller's narrowed token - onward."""
    DELEGATED.append({"agent": agent_name, "task": task, "token": user_token})
    return f"{agent_name} accepted"

USER_TOKEN = "obo:dana@corp:reports:read,reports:write"

print(f"{'agent':22s}{'in registry?':14s}{'approved?':11s}received work?")
for name in DISCOVERED:
    entry = REGISTRY.get(name)
    delegate(name, "summarise Q3 revenue", USER_TOKEN)      # admitted by discovery
    print(f"{name:22s}{str(bool(entry)):14s}"
          f"{str(bool(entry and entry['approved'])):11s}yes")

rogue = [d for d in DELEGATED if d["agent"] not in REGISTRY]
print(f"\nagents that received delegated work : {len(DELEGATED)}")
print(f"of which unregistered               : {len(rogue)}")
for r in rogue:
    print(f"   {r['agent']} now holds {r['token']}")
print()
print("It was admitted because it answered the protocol in the right place.")
print("It received the task AND the narrowed user token, so it can act as dana")
print("against every downstream that honours that token.")
assert rogue and all(r["token"] == USER_TOKEN for r in rogue)

## What you just proved

Three agents are discovered, two are in the registry, and all three receive delegated work — including the narrowed user token. The unregistered agent can now act as the requesting user against any downstream that honours it.

## Your turn

Ask how your orchestrator decides which agents may receive work. If the answer is a config list or service discovery, write down what would have to be true for an extra entry to be noticed.

---

**Next → [A1.12 · Cascading hallucination](https://spbreed.github.io/cyber-commons/lessons/A1.12.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.11.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.11.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*